# Ch 11. 벡터 RAG vs GraphRAG 직접 비교 (hands-on)

지식 그래프와 GraphRAG 스터디 · [study-graphrag](https://desty.github.io/study-graphrag/)

[Ch 11](https://desty.github.io/study-graphrag/part4/11-vector-vs-graph/) 본문. **설치 없이 Colab에서 바로 돈다** — 작은 코퍼스에 naive 벡터 검색(numpy 코사인)과 그래프 순회(networkx)를 나란히 돌려, 단순 사실/멀티홉 질문에서 **어디서 갈리는지** 직접 본다.

In [ ]:
!pip -q install networkx numpy
import numpy as np, networkx as nx

# 작은 코퍼스 (문장마다 사실 하나씩 흩어져 있음)
CORPUS = [
    "아스피린은 두통을 완화한다.",
    "아스피린은 아세틸살리실산을 성분으로 포함한다.",
    "부펜정은 아세틸살리실산을 성분으로 포함한다.",
    "이부프로펜은 두통을 완화한다.",
]

# 그래프용 트리플 (Ch 7에서 추출했다고 치자)
TRIPLES = [
    ("아스피린", "treats", "두통"), ("아스피린", "contains", "아세틸살리실산"),
    ("부펜정", "contains", "아세틸살리실산"), ("이부프로펜", "treats", "두통"),
]

## 벡터 RAG 베이스라인 — bag-of-words 코사인으로 비슷한 문장 top-k

In [ ]:
def tokenize(s):
    # 한국어는 조사 때문에 띄어쓰기 토큰이 잘 안 맞는다 → 문자 bigram으로 (아스피린 ~ 아스피린은 매칭)
    s = s.replace(".", "").replace(" ", "")
    return [s[i:i+2] for i in range(len(s) - 1)]

VOCAB = sorted({w for s in CORPUS for w in tokenize(s)})
def vec(s):
    v = np.zeros(len(VOCAB))
    for w in tokenize(s):
        if w in VOCAB: v[VOCAB.index(w)] += 1
    return v
DOC_VECS = np.array([vec(s) for s in CORPUS])

def vector_rag(q, k=2):
    qv = vec(q)
    sims = DOC_VECS @ qv / (np.linalg.norm(DOC_VECS, axis=1) * (np.linalg.norm(qv) + 1e-9) + 1e-9)
    top = sims.argsort()[::-1][:k]
    return [CORPUS[i] for i in top if sims[i] > 0]

## GraphRAG — networkx 그래프 순회

In [ ]:
G = nx.MultiDiGraph()
for s, r, o in TRIPLES:
    G.add_edge(s, o, rel=r)

def out_by(node, rel):
    return [v for _, v, d in G.out_edges(node, data=True) if d["rel"] == rel] if node in G else []
def in_by(node, rel):
    return [u for u, _, d in G.in_edges(node, data=True) if d["rel"] == rel] if node in G else []

def graph_same_ingredient(drug):
    """멀티홉: drug→(contains)→성분→(contains)←다른 약"""
    out = []
    for ing in out_by(drug, "contains"):
        out += [d for d in in_by(ing, "contains") if d != drug]
    return out

## 같은 질문 두 개를 두 시스템에 — 어디서 갈리나

In [ ]:
# ① 단순 사실: "아스피린의 성분은?"  → 한 문장에 답이 있다 (벡터 OK)
print("[factual] 아스피린의 성분은?")
print("  vector:", vector_rag("아스피린 성분"))
print("  graph :", out_by("아스피린", "contains"))

# ② 멀티홉: "아스피린과 같은 성분을 쓰는 다른 약은?"  → 두 문장을 잇는 추론 필요
print("\n[multihop] 아스피린과 같은 성분을 쓰는 다른 약은?")
print("  vector:", vector_rag("아스피린 같은 성분 다른 약"), "  ← '부펜정'을 직접 못 짚는다")
print("  graph :", graph_same_ingredient("아스피린"), "  ← 성분을 거쳐 연결")

## 무엇이 보이나

- **단순 사실**: 답이 한 문장에 있어 벡터로 충분하다. 그래프가 더 나을 게 없다.
- **멀티홉**: 벡터는 '아스피린'이 든 문장만 끌어올 뿐, *아세틸살리실산을 거쳐* '부펜정'으로 잇지 못한다. 그래프는 관계를 따라가 바로 답한다.

그래서 결론은 "둘 중 하나"가 아니라 **질문에 맞게 라우팅** — 단순 사실은 싼 벡터로, 멀티홉·전역은 그래프로. 본문 §4의 `router` 참고.

**연습**: 비용·지연 열을 더해 비교표를 채우고, 라우터를 붙여 단순 사실을 벡터로 보낼 때 비용이 얼마나 주는지 측정하라.